In [ ]:
# Colab setup (run this cell first)
# If you opened this notebook from GitHub in Colab, clone the repo so relative imports work.
# If the repo is PRIVATE, this clone will fail unless you provide a token.

import os
import sys

IN_COLAB = "COLAB_GPU" in os.environ or os.path.exists("/content")
REPO_URL = os.environ.get("REPO_URL", "https://github.com/momofahmi/NLP-sequence-classification")
REPO_DIR = os.environ.get("REPO_DIR", "/content/NLP-sequence-classification")

if IN_COLAB and not os.path.exists(REPO_DIR):
    try:
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
    except Exception as e:
        raise RuntimeError(
            "Could not clone the GitHub repo. If this repo is PRIVATE, Colab cannot access it.\n\n"
            "Fix options:\n"
            "- Make the repo public (recommended for lecturers).\n"
            "- OR set REPO_URL to include a GitHub token, e.g.:\n"
            "  https://<TOKEN>@github.com/momofahmi/NLP-sequence-classification\n"
            "  (not recommended for sharing).\n"
        ) from e

if IN_COLAB:
    %cd {REPO_DIR}
    get_ipython().system("pip -q install -r requirements.txt")

# Ensure repo root importable
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("✅ Setup complete")


In [ ]:
# Notebook run mode
# - DEMO_MODE=True: safe for "Runtime -> Run all" (fast, small subset)
# - DEMO_MODE=False: full experiment (slow; use GPU and expect long runtime)

import os

DEMO_MODE = os.environ.get("DEMO_MODE", "1") == "1"

# Common settings
TASK = os.environ.get("TASK", "sarcasm")  # sarcasm recommended for cross-variety
MODEL_NAME = os.environ.get("MODEL_NAME", "roberta-base")
MAX_LEN = int(os.environ.get("MAX_LEN", "128"))

# Demo defaults (fast)
if DEMO_MODE:
    SEEDS = [42]
    NUM_EPOCHS = 1
    TRAIN_BS = 8
    EVAL_BS = 16
    MAX_TRAIN_EX = 500
    MAX_VAL_EX = 200
    MAX_TEST_EX = 300
    MAX_TRAIN_CONDITIONS = 1
    MAX_TEST_SETS = 3
else:
    SEEDS = [42, 123]
    NUM_EPOCHS = 3
    TRAIN_BS = 16
    EVAL_BS = 32
    MAX_TRAIN_EX = None
    MAX_VAL_EX = None
    MAX_TEST_EX = None
    MAX_TRAIN_CONDITIONS = None
    MAX_TEST_SETS = None

print(
    "Mode:",
    "DEMO" if DEMO_MODE else "FULL",
    "| task=", TASK,
    "| model=", MODEL_NAME,
)


# Check dataset columns

In [1]:
import os
import sys

# Ensure project root is importable (works on Colab and locally)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.besstie_data_loader import load_besstie, get_train_conditions, get_test_conditions

ds = load_besstie()
print(ds["train"][0])

c:\Users\joela\Documents\code\ai\nlp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'text': "I'm a member of the Green Party but I'll be voting Lib Dem as it's so tight here between Lib Dem and Tory. I cannot contemplate our useless tit of a Tory MP being reelected. I'll use the Swap My Vote website so someone somewhere can vote Green for me.", 'variety': 'en-UK', 'source': 'Reddit', 'Sentiment': 0.0, 'Sarcasm': 0.0}


In [18]:
from datasets import Dataset

from transformers import (
    RobertaForSequenceClassification,
    RobertaTokenizer,
    TrainingArguments,
    Trainer,
)

import torch
import numpy as np

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)

LABEL_COL = "Sarcasm" if TASK.lower() == "sarcasm" else "Sentiment"
TEXT_COL = "text"

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

# Function to tokenize each batch of the dataset
def tokenize(batch):
    tokens = tokenizer(
        batch[TEXT_COL],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )
    tokens["labels"] = [int(label) for label in batch[LABEL_COL]]

    return tokens

def prepare_dataset(dataset):
    """Tokenizes and formats dataset for trainer"""

    tokenized = dataset.map(tokenize, batched=True)
    tokenized = tokenized.remove_columns(
        [c for c in tokenized.column_names if c not in
        ["input_ids", "attention_mask", "labels"]]
    )
    
    tokenized.set_format("torch")
    return tokenized

In [19]:
# Functions to compute model metrics
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
)


def compute_metrics(eval_pred):
    """Metrics used by Trainer during eval."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "accuracy": (predictions == labels).mean(),
    }


def _target_names(label_col: str):
    if label_col.lower() == "sarcasm":
        return ["Not Sarcastic", "Sarcastic"]
    if label_col.lower() == "sentiment":
        return ["Negative", "Positive"]
    return ["Class 0", "Class 1"]


def full_evaluation(y_true, y_pred, label_col: str):
    """Full metrics for report — called after training completes."""
    names = _target_names(label_col)
    return {
        "macro_f1": round(f1_score(y_true, y_pred, average="macro"), 4),
        "precision": round(precision_score(y_true, y_pred, average="macro"), 4),
        "recall": round(recall_score(y_true, y_pred, average="macro"), 4),
        "per_class_f1": f1_score(y_true, y_pred, average=None).tolist(),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
        "report": classification_report(y_true, y_pred, target_names=names),
    }

In [20]:
# Training function - using Trainer

def train_roberta(train_data, val_data, seed=42, output_dir="./tmp"):
    """Fine-tunes RoBERTa on given training data."""

    torch.manual_seed(seed)
    np.random.seed(seed)

    model = RobertaForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
    )

    # Tokenize training/validation
    train_tokenized = prepare_dataset(train_data)
    val_tokenized = prepare_dataset(val_data)

    import inspect

    eval_key = (
        "eval_strategy"
        if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters
        else "evaluation_strategy"
    )

    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=TRAIN_BS,
        per_device_eval_batch_size=EVAL_BS,
        learning_rate=2e-5,
        **{eval_key: "epoch"},
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        seed=seed,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    return model, tokenizer

def evaluate_on_testset(model, test_data, label_col: str):
    """Runs trained model on a test set."""

    test_tokenized = prepare_dataset(test_data)

    trainer = Trainer(model=model)
    output = trainer.predict(test_tokenized)

    y_pred = np.argmax(output.predictions, axis=1)
    y_true = test_tokenized["labels"].numpy()

    return full_evaluation(y_true, y_pred, label_col=label_col)

In [21]:
print("Train columns:", ds["train"].column_names)

Train columns: ['text', 'variety', 'source', 'Sentiment', 'Sarcasm']


In [ ]:
# Experiment loop
from tqdm import tqdm
import json

os.makedirs("results", exist_ok=True)
os.makedirs("figures", exist_ok=True)

all_results = {}

train_conditions = get_train_conditions(ds)
test_sets = get_test_conditions(ds)
val_data = ds["validation"]

# DEMO_MODE: keep run-all fast + deterministic
if MAX_TRAIN_CONDITIONS is not None:
    train_conditions = dict(list(train_conditions.items())[:MAX_TRAIN_CONDITIONS])
if MAX_TEST_SETS is not None:
    test_sets = dict(list(test_sets.items())[:MAX_TEST_SETS])

if MAX_VAL_EX is not None:
    val_data = val_data.select(range(min(MAX_VAL_EX, len(val_data))))

print("Train conditions:", list(train_conditions.keys()))
print("Test sets:", list(test_sets.keys()))
print("Seeds:", SEEDS)

# Training conditions
for condition_name, train_data in tqdm(
    train_conditions.items(),
    desc="Conditions",
    position=0,
):
    if MAX_TRAIN_EX is not None:
        train_data = train_data.select(range(min(MAX_TRAIN_EX, len(train_data))))

    condition_results = {}

    # Loop for the seeds
    for seed in tqdm(
        SEEDS,
        desc=f" {condition_name}",
        position=1,
        leave=False,
    ):
        model, tok = train_roberta(
            train_data,
            val_data,
            seed=seed,
            output_dir=f"./tmp/{condition_name}_seed{seed}",
        )

        seed_results = {}
        for test_name, test_data in tqdm(
            test_sets.items(),
            desc=" Evaluating",
            position=2,
            leave=False,
        ):
            if MAX_TEST_EX is not None:
                test_data = test_data.select(range(min(MAX_TEST_EX, len(test_data))))

            results = evaluate_on_testset(model, test_data, label_col=LABEL_COL)
            seed_results[test_name] = results

        condition_results[f"seed_{seed}"] = seed_results

    averaged = {}

    for test_name in test_sets.keys():
        f1_scores = [
            condition_results[f"seed_{s}"][test_name]["macro_f1"]
            for s in SEEDS
        ]
        averaged[test_name] = {
            "macro_f1_mean": round(float(np.mean(f1_scores)), 4),
            "macro_f1_std": round(float(np.std(f1_scores)), 4),
        }

    all_results[condition_name] = {"by_seed": condition_results, "averaged": averaged}
    with open(f"results/{condition_name}.json", "w") as f:
        json.dump(all_results[condition_name], f, indent=2)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4871.49it/s]
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map: 100%|██████████| 1203/1203 [00:00<00

In [ ]:
# Results Visualisation (incl cross variety matrix)

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

conditions = list(train_conditions.keys())
test_names = list(test_sets.keys())

matrix = np.array([
    [all_results[c]["averaged"][t]["macro_f1_mean"] for t in test_names]
    for c in conditions
])

plt.figure(figsize=(8, 6))

sns.heatmap(
    matrix,
    annot=True,
    fmt=".3f",
    xticklabels=["Test UK", "Test AU", "Test IN"],
    yticklabels=["UK only", "AU only", "IN only", "Inner pool", "All pool"],
    cmap="YlOrRd",
    vmin=0.5,
    vmax=1.0,
)
plt.title("Cross-Variety Evaluation Matrix — Macro-F1 (RoBERTa)")
plt.ylabel("Trained on")
plt.xlabel("Tested on")
plt.tight_layout()
plt.savefig("figures/cross_variety_matrix.png", dpi=150)
plt.show()

results_df = pd.DataFrame(
    matrix,
    index=["UK only", "AU only", "IN only", "Inner pool", "All pool"],
    columns=["Test UK", "Test AU", "Test IN"],
)
print("\nCross-Variety Matrix:")
print(results_df.to_string())

In [ ]:
# Find confusion matrix for best conditions
best_condition = max(
    conditions,
    key=lambda c: np.mean([
        all_results[c]["averaged"][t]["macro_f1_mean"]
        for t in test_names
    ])
)
print(f"Best condition: {best_condition}")

best_cm = np.array(
    all_results[best_condition]["by_seed"]["seed_42"]["uk_test"]["confusion_matrix"]
)

plt.figure(figsize=(6, 5))
sns.heatmap(
    best_cm,
    annot=True,
    fmt="d",
    xticklabels=["Not Sarcastic", "Sarcastic"],
    yticklabels=["Not Sarcastic", "Sarcastic"],
    cmap="Blues"
)
plt.title(f"Confusion Matrix — {best_condition} → UK test")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.savefig("figures/confusion_matrix_best.png", dpi=150)
plt.show()